# Lab 5.2 - Heterogeneous Graph Learning

This notebook follows the PyG tutorial "Heterogeneous Graph Learning".

Since the tutorial does not end with exercises, the goal here is to present the main ideas and run representative code examples:
- create a heterogeneous graph with multiple node and edge types;
- inspect metadata and typed dictionaries;
- apply heterogeneous transforms;
- convert a homogeneous GNN with `to_hetero()`;
- build a custom typed model with `HeteroConv`.

In [1]:
import copy
import torch
import torch.nn.functional as F
import torch_geometric
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero, HeteroConv, GCNConv, GATConv, Linear

print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)

d:\Documents\FMI\Master Anul II\EDDL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cu128
torch_geometric: 2.7.0


## 1. What is a heterogeneous graph?

A homogeneous graph has one node type and one edge type. A heterogeneous graph has multiple kinds of entities and relations.

Example:
- nodes: `paper`, `author`, `institution`;
- edges: `author -> writes -> paper`, `author -> affiliated_with -> institution`, `paper -> cites -> paper`.

This means we cannot store the whole graph in a single `x` tensor and a single adjacency relation. PyG uses `HeteroData` so every node type and edge type can keep its own tensors.

In [2]:
torch.manual_seed(0)

data = HeteroData()

data['paper'].x = torch.tensor([
    [1.0, 0.0, 0.0, 1.0],
    [0.8, 0.2, 0.1, 0.9],
    [0.0, 1.0, 1.0, 0.0],
    [0.1, 0.9, 0.8, 0.2],
], dtype=torch.float)
data['paper'].y = torch.tensor([0, 0, 1, 1], dtype=torch.long)

data['author'].x = torch.tensor([
    [1.0, 1.0, 0.0, 0.0],
    [0.9, 0.8, 0.1, 0.0],
    [0.0, 0.1, 1.0, 1.0],
], dtype=torch.float)

data['institution'].x = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
], dtype=torch.float)

data['author', 'writes', 'paper'].edge_index = torch.tensor([
    [0, 0, 1, 2],
    [0, 1, 2, 3],
], dtype=torch.long)

data['author', 'affiliated_with', 'institution'].edge_index = torch.tensor([
    [0, 1, 2],
    [0, 0, 1],
], dtype=torch.long)

data['paper', 'cites', 'paper'].edge_index = torch.tensor([
    [0, 1, 2, 3],
    [1, 0, 3, 2],
], dtype=torch.long)

print(data)

HeteroData(
  paper={
    x=[4, 4],
    y=[4],
  },
  author={ x=[3, 4] },
  institution={ x=[2, 4] },
  (author, writes, paper)={ edge_index=[2, 4] },
  (author, affiliated_with, institution)={ edge_index=[2, 3] },
  (paper, cites, paper)={ edge_index=[2, 4] }
)


In [3]:
print('Node types:', data.node_types)
print('Edge types:', data.edge_types)
print('Metadata:', data.metadata())
print('x_dict keys:', list(data.x_dict.keys()))
print('edge_index_dict keys:', list(data.edge_index_dict.keys()))

homogeneous = data.to_homogeneous()
print('Homogeneous view:')
print(homogeneous)

Node types: ['paper', 'author', 'institution']
Edge types: [('author', 'writes', 'paper'), ('author', 'affiliated_with', 'institution'), ('paper', 'cites', 'paper')]
Metadata: (['paper', 'author', 'institution'], [('author', 'writes', 'paper'), ('author', 'affiliated_with', 'institution'), ('paper', 'cites', 'paper')])
x_dict keys: ['paper', 'author', 'institution']
edge_index_dict keys: [('author', 'writes', 'paper'), ('author', 'affiliated_with', 'institution'), ('paper', 'cites', 'paper')]
Homogeneous view:
Data(edge_index=[2, 11], x=[9, 4], y=[9], node_type=[9], edge_type=[11])


Key takeaway: `HeteroData` stores tensors by type, and the model later consumes dictionaries such as `x_dict` and `edge_index_dict`.

`to_homogeneous()` is useful for inspection because it merges everything into a single typed graph, but the type information is still tracked internally.

In [4]:
data_t = T.ToUndirected()(copy.deepcopy(data))
data_t = T.AddSelfLoops()(data_t)
data_t = T.NormalizeFeatures()(data_t)

print('Edge types after ToUndirected/AddSelfLoops:')
for edge_type in data_t.edge_types:
    print(' ', edge_type, data_t[edge_type].edge_index.shape)

print('Row-normalized paper features:')
print(data_t['paper'].x)

Edge types after ToUndirected/AddSelfLoops:
  ('author', 'writes', 'paper') torch.Size([2, 4])
  ('author', 'affiliated_with', 'institution') torch.Size([2, 3])
  ('paper', 'cites', 'paper') torch.Size([2, 8])
  ('paper', 'rev_writes', 'author') torch.Size([2, 4])
  ('institution', 'rev_affiliated_with', 'author') torch.Size([2, 3])
Row-normalized paper features:
tensor([[0.5000, 0.0000, 0.0000, 0.5000],
        [0.4000, 0.1000, 0.0500, 0.4500],
        [0.0000, 0.5000, 0.5000, 0.0000],
        [0.0500, 0.4500, 0.4000, 0.1000]])


These transforms illustrate three common preprocessing steps from the tutorial:
- `ToUndirected()` adds reverse edge types like `rev_writes`;
- `AddSelfLoops()` adds self-information where this makes sense;
- `NormalizeFeatures()` rescales node features.

## 2. Converting a homogeneous model with `to_hetero()`

The easiest way to enter heterogeneous learning is to write a normal homogeneous GNN and let PyG duplicate its message-passing logic for every edge type.

In [5]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = GNN(hidden_channels=8, out_channels=2)
model = to_hetero(model, data_t.metadata(), aggr='sum')

with torch.no_grad():
    out_dict = model(data_t.x_dict, data_t.edge_index_dict)

print('Output keys:', list(out_dict.keys()))
for key, value in out_dict.items():
    print(key, value.shape)

Output keys: ['paper', 'author', 'institution']
paper torch.Size([4, 2])
author torch.Size([3, 2])
institution torch.Size([2, 2])


In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
history = []

for epoch in range(1, 101):
    optimizer.zero_grad()
    out_dict = model(data_t.x_dict, data_t.edge_index_dict)
    loss = F.cross_entropy(out_dict['paper'], data_t['paper'].y)
    loss.backward()
    optimizer.step()
    if epoch in {1, 20, 50, 100}:
        history.append((epoch, float(loss)))

with torch.no_grad():
    out_dict = model(data_t.x_dict, data_t.edge_index_dict)
    pred = out_dict['paper'].argmax(dim=-1)
    acc = (pred == data_t['paper'].y).float().mean().item()

print('Loss history:')
for epoch, loss_value in history:
    print(f'  epoch={epoch:03d}, loss={loss_value:.4f}')
print('Final paper predictions:', pred.tolist())
print('Paper labels:           ', data_t['paper'].y.tolist())
print('Final accuracy on paper nodes:', acc)

C:\Users\Alex\AppData\Local\Temp\ipykernel_47940\3448408292.py:11: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  history.append((epoch, float(loss)))


Loss history:
  epoch=001, loss=0.8250
  epoch=020, loss=0.0000
  epoch=050, loss=0.0000
  epoch=100, loss=0.0000
Final paper predictions: [0, 0, 1, 1]
Paper labels:            [0, 0, 1, 1]
Final accuracy on paper nodes: 1.0


On this tiny graph the converted model overfits easily, which is fine here: the point is not to benchmark performance, but to show that `to_hetero()` really turns a homogeneous GNN into a typed GNN that outputs one tensor per node type.

## 3. Building a custom heterogeneous model with `HeteroConv`

`to_hetero()` uses the same operator for every relation. `HeteroConv` gives more control: we can choose a different operator for each edge type.

In [7]:
class TinyHeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv = HeteroConv({
            ('paper', 'cites', 'paper'): GCNConv(-1, hidden_channels),
            ('author', 'writes', 'paper'): SAGEConv((-1, -1), hidden_channels),
            ('paper', 'rev_writes', 'author'): GATConv((-1, -1), hidden_channels, add_self_loops=False),
            ('author', 'affiliated_with', 'institution'): SAGEConv((-1, -1), hidden_channels),
            ('institution', 'rev_affiliated_with', 'author'): SAGEConv((-1, -1), hidden_channels),
        }, aggr='sum')
        self.lin = Linear(-1, out_channels)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv(x_dict, edge_index_dict)
        x_dict = {key: value.relu() for key, value in x_dict.items()}
        return {key: self.lin(value) for key, value in x_dict.items()}

hetero_model = TinyHeteroGNN(hidden_channels=8, out_channels=2)
with torch.no_grad():
    hetero_out = hetero_model(data_t.x_dict, data_t.edge_index_dict)

print('HeteroConv output keys:', list(hetero_out.keys()))
for key, value in hetero_out.items():
    print(key, value.shape)

HeteroConv output keys: ['paper', 'author', 'institution']
paper torch.Size([4, 2])
author torch.Size([3, 2])
institution torch.Size([2, 2])


## 4. Final takeaway

The tutorial presents three main ways to work with heterogeneous graphs:
- represent them with `HeteroData`;
- convert a homogeneous model automatically with `to_hetero()`;
- build a custom typed model with `HeteroConv`;
- or use operators designed specifically for heterogeneous graphs, such as HGT-style layers.

In this notebook, the code was kept lightweight on purpose, but it still demonstrates the main idea: message passing is now conditioned on relation type, not just on graph structure.